In [13]:
import pandas as pd
import anthropic
import json

## Arthouse
Arthouse cinema refers to films that prioritize artistic expression, formal experimentation, or thematic depth over commercial appeal. They tend to feature non-linear or ambiguous narratives, a strong directorial voice, and subjects rooted in psychological, philosophical, or social themes rather than genre formulas. They're typically independently produced, circulated through festival circuits, and aimed at an engaged audience seeking provocation or reflection rather than conventional entertainment.

In [4]:
df = pd.read_csv("films_enriched.csv")
df_films = pd.read_csv("films_decoded.csv")

In [12]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [14]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(df_films.iloc[0])

titleId                                                         tt1821427
imdbUrl                             https://www.imdb.com/title/tt1821427/
originalTitle                                                    Offender
titleType                                                           movie
releaseYear                                                        2012.0
runtimeMinutes                                                      102.0
isAdult                                                             False
imdbRating                                                            6.0
numberOfVotes                                                      4994.0
allCountries                                                           GB
mainCountry                                                            GB
allLanguages                                                           en
firstLanguage                                                          en
englishTitle                          

In [7]:
df_films.columns

Index(['titleId', 'imdbUrl', 'originalTitle', 'titleType', 'releaseYear',
       'runtimeMinutes', 'isAdult', 'imdbRating', 'numberOfVotes',
       'allCountries', 'mainCountry', 'allLanguages', 'firstLanguage',
       'englishTitle', 'topFiveActors', 'directors', 'writers', 'plotShort',
       'plotMedium', 'plotLong', 'genres', 'keywords', 'production', 'tmdb_id',
       'budget', 'revenue', 'tmdb_popularity', 'tmdb_vote_average',
       'tmdb_vote_count', 'original_language', 'tagline', 'status',
       'spoken_languages', 'production_countries', 'aggregateRating',
       'totalVotes', 'rating_1', 'rating_2', 'rating_3', 'rating_4',
       'rating_5', 'rating_6', 'rating_7', 'rating_8', 'rating_9', 'rating_10',
       'ml_rating_count', 'ml_rating_mean', 'ml_rating_std',
       'ml_rating_median', 'ml_rating_0_5', 'ml_rating_1_0', 'ml_rating_1_5',
       'ml_rating_2_0', 'ml_rating_2_5', 'ml_rating_3_0', 'ml_rating_3_5',
       'ml_rating_4_0', 'ml_rating_4_5', 'ml_rating_5_0', 'ml_

In [2]:
df_scored = pd.read_csv("films_scored.csv")

/var/folders/rs/xp_jvmcs6yxft09g6r_s670m0000gn/T/ipykernel_9937/3437574264.py:1: DtypeWarning: Columns (62) have mixed types. Specify dtype option on import or set low_memory=False.
  df_scored = pd.read_csv("films_scored.csv")


In [5]:
df_scored = df_scored.dropna(subset=["arthouse_reasoning"])

In [15]:
json = json.loads(df_scored["arthouse_reasoning"].iloc[0])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [19]:
raw = df_scored["arthouse_reasoning"].iloc[0]

clean = raw.replace("JSON parse error:", "").replace("```json", "").replace("```", "").strip()

data = json.loads(clean)

JSONDecodeError: Unterminated string starting at: line 1 column 27 (char 26)

In [20]:
df_jsonl = pd.read_json("msgbatch_019HMReAtCghGEtFVD7EpwLz_results.jsonl", lines=True)
print(df_jsonl.head())

    custom_id                                             result
0   tt1821427  {'type': 'succeeded', 'message': {'model': 'cl...
1  tt23460232  {'type': 'succeeded', 'message': {'model': 'cl...
2   tt1320100  {'type': 'succeeded', 'message': {'model': 'cl...
3  tt26348541  {'type': 'succeeded', 'message': {'model': 'cl...
4   tt0362277  {'type': 'succeeded', 'message': {'model': 'cl...


In [27]:
json_entry = df_jsonl.iloc[0].to_json()
parsed = json.loads(json_entry)

print(json.dumps(parsed, indent=2))

{
  "custom_id": "tt1821427",
  "result": {
    "type": "succeeded",
    "message": {
      "model": "claude-haiku-4-5-20251001",
      "id": "msg_01E2ZFSgssEcMKeiRB7CTZgm",
      "type": "message",
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "```json\n{\"score\": 6, \"reasoning\": \"A British crime drama with social commentary on the justice system that balances indie sensibilities and serious thematic concerns with accessibility, though limited information suggests it may lean slightly more toward conventional drama than experimental arthouse.\"}\n```"
        }
      ],
      "stop_reason": "end_turn",
      "stop_sequence": null,
      "stop_details": null,
      "usage": {
        "input_tokens": 370,
        "cache_creation_input_tokens": 0,
        "cache_read_input_tokens": 0,
        "cache_creation": {
          "ephemeral_5m_input_tokens": 0,
          "ephemeral_1h_input_tokens": 0
        },
        "output_tokens": 6

In [34]:
import pandas as pd
import json
import re

# --- Load results ---
df_results = pd.read_json("msgbatch_019HMReAtCghGEtFVD7EpwLz_results.jsonl", lines=True)
print(f"{len(df_results)} results loaded")

# --- Parse scores ---
records = []
tokens = 0
input_tokens = 0
output_tokens = 0

for _, row in df_results.iterrows():
    title_id = row["custom_id"]
    result = row["result"]

    if isinstance(result, str):
        result = json.loads(result)

    if result["type"] != "succeeded":
        records.append({"titleId": title_id, "arthouse_score": None, "arthouse_reasoning": result["type"]})
        continue

    text = result["message"]["content"][0]["text"]

    tokens += (int(result["message"]["usage"]["input_tokens"]) + int(result["message"]["usage"]["output_tokens"]))
    input_tokens += int(result["message"]["usage"]["input_tokens"])
    output_tokens += int(result["message"]["usage"]["output_tokens"])

    #print(result["message"]["usage"]["output_tokens"])
    
    # Strip markdown code fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text.strip())

    try:
        parsed = json.loads(text)
        records.append({
            "titleId": title_id,
            "arthouse_score": parsed.get("score"),
            "arthouse_reasoning": parsed.get("reasoning"),
        })
    except json.JSONDecodeError as e:
        records.append({"titleId": title_id, "arthouse_score": None, "arthouse_reasoning": f"Parse error: {e}"})

print("total tokens: ", tokens)
print("avg token use: ", tokens/50)

print("in total tokens: ", input_tokens)
print("avg in token use: ", input_tokens/50)

print("out total tokens: ", output_tokens)
print("avg out token use: ", output_tokens/50)

scores_df = pd.DataFrame(records)
print(f"Scored: {scores_df['arthouse_score'].notna().sum()}  |  Failed: {scores_df['arthouse_score'].isna().sum()}")

# --- Merge with original films ---
# df_films = pd.read_csv("films_decoded.csv", dtype={"titleId": str})
# scores_df["titleId"] = scores_df["titleId"].astype(str)
# df_merged = df_films.merge(scores_df, on="titleId", how="left")

# # --- Quick stats ---
# print(f"Mean score: {df_merged['arthouse_score'].mean():.2f}")
# df_merged["arthouse_score"].value_counts().sort_index()

50 results loaded
total tokens:  22508
avg token use:  450.16
in total tokens:  19094
avg in token use:  381.88
out total tokens:  3414
avg out token use:  68.28
Scored: 50  |  Failed: 0


In [33]:
450*50000

22500000

In [29]:
df_merged

,titleId,imdbUrl,originalTitle,titleType,releaseYear,runtimeMinutes,isAdult,imdbRating,numberOfVotes,allCountries,...,ml_rating_2_0,ml_rating_2_5,ml_rating_3_0,ml_rating_3_5,ml_rating_4_0,ml_rating_4_5,ml_rating_5_0,ml_tags,arthouse_score,arthouse_reasoning
0,tt1821427,https://www.imdb.com/title/tt1821427/,Offender,movie,2012.0,102.0,False,6.0,4994.0,GB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,A British crime drama with social commentary o...
1,tt23460232,https://www.imdb.com/title/tt23460232/,Bijuterie,movie,NaN,NaN,False,NaN,NaN,LT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,This Lithuanian film exhibits strong arthouse ...
2,tt1320100,https://www.imdb.com/title/tt1320100/,Kato apo to makigiaz sou,movie,2008.0,90.0,False,4.8,40.0,GR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,This Greek drama explores psychological trauma...
3,tt26348541,https://www.imdb.com/title/tt26348541/,La empresa,movie,2023.0,94.0,False,6.4,5.0,DE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,This documentary employs a meta-theatrical app...
4,tt0362277,https://www.imdb.com/title/tt0362277/,Der Verräter,movie,1917.0,NaN,False,NaN,NaN,DE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,As a 1917 German silent film in the crime-dram...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,tt0170019,https://www.imdb.com/title/tt0170019/,The Hypocrites,movie,1923.0,NaN,False,NaN,NaN,"GB, NL",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49996,tt30487560,https://www.imdb.com/title/tt30487560/,Copeland,movie,NaN,NaN,False,NaN,NaN,ES,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49997,tt0081536,https://www.imdb.com/title/tt0081536/,Sono fotogenico,movie,1980.0,114.0,False,6.1,557.0,"IT, FR",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49998,tt17277130,https://www.imdb.com/title/tt17277130/,Le Requiem de Wallenberg,movie,1999.0,84.0,False,NaN,NaN,FR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
